# Training and validation overlap audit

## Load and check the two subsets

Read the saved seed-42 training and validation CSVs using the parsing options from `02_eda_restart.ipynb`. Verify that both contain the same 46 numeric feature columns, finite values, and an original label for every row.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.api.types import is_numeric_dtype

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

subset_dir = repo_root / "data/subsets"
paths = {
    "Training": subset_dir / "ciciot2023_train_stratified_100000_seed42.csv",
    "Validation": subset_dir / "ciciot2023_validation_stratified_25000_seed42.csv",
}
frames = {
    split: pd.read_csv(path, low_memory=False, skip_blank_lines=False)
    for split, path in paths.items()
}
feature_columns = frames["Training"].columns.drop("label")
if len(feature_columns) != 46:
    raise ValueError(f"Expected 46 feature columns, found {len(feature_columns)}.")

for split, frame in frames.items():
    if not frame.columns.drop("label").equals(feature_columns):
        raise ValueError(f"{split} feature columns differ from the training schema.")
    features = frame[feature_columns]
    if not all(map(is_numeric_dtype, features.dtypes)):
        raise ValueError(f"{split} has nonnumeric feature columns.")
    if not np.isfinite(features.to_numpy()).all() or frame["label"].isna().any():
        raise ValueError(f"{split} has nonfinite features or missing labels.")

display(pd.DataFrame([
    {"Split": split, "Rows": len(frame), "Numeric features": len(feature_columns)}
    for split, frame in frames.items()
]).set_index("Split"))

,Rows,Numeric features
Split,,
Training,100000,46
Validation,25000,46


**Interpretation.** The two inputs passed the schema and value checks, so every row can be compared using the same 46 features.

## Compare exact feature patterns

Reuse the EDA notebook’s tuple-key approach. Compare parsed numeric values directly, without rounding, and keep original labels in counters outside the feature key. For each shared key, sum its validation counts once regardless of how many training rows have that key. Classify `BenignTraffic` as benign and the other original CICIoT2023 labels as attack.

In [2]:
patterns_by_split = {}
for split, frame in frames.items():
    patterns = {}
    for row in frame[[*feature_columns, "label"]].itertuples(index=False, name=None):
        feature_key, label = row[:-1], row[-1]
        patterns.setdefault(feature_key, Counter())[label] += 1
    patterns_by_split[split] = patterns

training = patterns_by_split["Training"]
validation = patterns_by_split["Validation"]
shared = training.keys() & validation.keys()

matching_labels = Counter()
for key in shared:
    matching_labels.update(validation[key])

matching_rows = matching_labels.total()
validation_rows = len(frames["Validation"])
benign_rows = matching_labels["BenignTraffic"]
attack_rows = matching_rows - benign_rows
conflicting_patterns = sum(
    len(training[key].keys() | validation[key].keys()) > 1 for key in shared
)
reproduced = (len(shared), matching_rows) == (25, 25)

summary = pd.Series({
    "Distinct shared feature patterns": len(shared),
    "Matching validation rows": matching_rows,
    "Matching validation rows (%)": f"{100 * matching_rows / validation_rows:.4f}%",
    "Benign matching validation rows": benign_rows,
    "Attack matching validation rows": attack_rows,
    "Shared patterns with conflicting original labels": conflicting_patterns,
    "Any shared pattern with conflicting original labels": "Yes" if conflicting_patterns else "No",
    "Earlier result reproduced (25 patterns and 25 validation rows)": "Yes" if reproduced else "No",
}, name="Result")
display(summary.to_frame())

,Result
Distinct shared feature patterns,25
Matching validation rows,25
Matching validation rows (%),0.1000%
Benign matching validation rows,2
Attack matching validation rows,23
Shared patterns with conflicting original labels,0
Any shared pattern with conflicting original labels,No
Earlier result reproduced (25 patterns and 25 validation rows),Yes


**Interpretation.** 25 of 25,000 validation rows (0.1000%) have the same 46 feature values as at least one training row. These rows span 25 distinct shared patterns and include 2 benign and 23 attack rows. The original labels agree for every shared pattern. This reproduces the earlier EDA result of 25 shared patterns and 25 matching validation rows.